# IOAI — 2026 Summer Online Production Qc (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2026/nyari-online/B/adatok >/dev/null && git checkout -q
import shutil, glob, os
for f in glob.glob('haio/2026/nyari-online/B/adatok/*.csv'):
    if os.path.basename(f) != 'solution.csv': shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Production Line Quality Control (Gyártósori Minőségellenőrzés)

> **HAIO 2026 — Summer Online Qualifier, Task B (ML)**

Predict the probability that each manufactured item is defective (`Selejt`). Score = **ROC-AUC**.
Semi-supervised: only **150 labelled** rows (`train_labeled.csv`) + **12 850 unlabelled** (`train_unlabeled.csv`); predict `test.csv` (2 000). The hidden test labels are bundled as the grading key — **Submit** `submission.csv` (`ID,Selejt` probability) → ROC-AUC.

Baseline: HistGradientBoosting on the labelled rows. (Improve by using the unlabelled data — self-training / pseudo-labels.)

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
def prep(Xtr, Xte):
    cat = [c for c in Xtr.columns if Xtr[c].dtype == 'object']
    num = [c for c in Xtr.columns if c not in cat]
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    Atr = Xtr[cat].astype('object').where(Xtr[cat].notna(), 'NA')
    Ate = Xte[cat].astype('object').where(Xte[cat].notna(), 'NA')
    Ctr = enc.fit_transform(Atr); Cte = enc.transform(Ate)
    import numpy as np
    return (np.hstack([Xtr[num].to_numpy(float), Ctr]),
            np.hstack([Xte[num].to_numpy(float), Cte]))

train = pd.read_csv('train_labeled.csv')
test = pd.read_csv('test.csv')
y = train['Selejt'].to_numpy()
Xtr_df = train.drop(columns=['ID', 'Selejt'])
Xte_df = test.drop(columns=['ID'])
Xtr, Xte = prep(Xtr_df, Xte_df)
Xtr.shape, Xte.shape, y.mean()

## Self-check (5-fold CV AUC on the 150 labelled rows), then fit on all + predict test

In [ ]:
clf = HistGradientBoostingClassifier(random_state=42)
cv = cross_val_score(clf, Xtr, y, cv=5, scoring='roc_auc')
print(f'CV ROC-AUC: {cv.mean():.4f} +/- {cv.std():.4f}')

In [ ]:
clf = HistGradientBoostingClassifier(random_state=42).fit(Xtr, y)
proba = clf.predict_proba(Xte)[:, 1]
pd.DataFrame({'ID': test['ID'], 'Selejt': proba}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(test))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)